In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from sklearn.decomposition import PCA
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image_cc_3.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_3.jpg
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg

# Lecture 17 - Representation and Self-Supervised Learning

What do self-supervised features look like? We visualise DINOv2 patch features, use them to find correspondences between very different views, and look at two vision-language models trained contrastively: CLIP (softmax over a batch) and SigLIP (sigmoid per pair). Finally, CLIP *reads*.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
def load_rgb(filename, max_side=None):
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    if max_side:
        s = max_side / max(image.shape[:2])
        image = cv2.resize(image, (0, 0), fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return image

images = {name: load_rgb(f'{name}.jpg', 1280) for name in ['image_dog', 'image_me', 'image_cc_1', 'image_cc_3', 'image']}

## DINOv2 patch features

DINOv2 is a ViT trained without labels (self-distillation). Every 14x14 patch gets a 768-dimensional feature. For each image, we project the patch features onto their first three principal components and show them as RGB. Without ever seeing a label or a mask, the features separate objects from the background and group their parts.

In [ ]:
from transformers import AutoModel
dino = AutoModel.from_pretrained('facebook/dinov2-base').to(device).eval()
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def dino_features(image, patches_long_side=64):
    # resize so that both sides are multiples of the patch size (14)
    h, w = image.shape[:2]
    s = patches_long_side * 14 / max(h, w)
    ph, pw = int(round(h * s / 14)), int(round(w * s / 14))
    x = cv2.resize(image, (pw * 14, ph * 14), interpolation=cv2.INTER_AREA) / 255.0
    x = torch.from_numpy((x - mean) / std).permute(2, 0, 1)[None].float().to(device)
    with torch.no_grad():
        tokens = dino(pixel_values=x).last_hidden_state[0, 1:]     # drop the class token
    return F.normalize(tokens, dim=-1).cpu().numpy().reshape(ph, pw, -1)

features = {name: dino_features(im) for name, im in images.items()}

def pca_rgb(f):
    patches = f.reshape(-1, f.shape[-1])
    projected = PCA(n_components=3).fit_transform(patches)
    lo, hi = np.percentile(projected, 1, axis=0), np.percentile(projected, 99, axis=0)
    return np.clip((projected - lo) / (hi - lo), 0, 1).reshape(*f.shape[:2], 3)

names = ['image_dog', 'image_me', 'image_cc_1', 'image']
fig, axs = plt.subplots(2, len(names), figsize=(6 * len(names), 8))
for col, name in enumerate(names):
    f = features[name]
    rgb = pca_rgb(f)
    axs[0, col].imshow(images[name])
    axs[1, col].imshow(rgb, interpolation='nearest')
    axs[1, col].set_title(f'{f.shape[0]}x{f.shape[1]} patch features')
    for ax in axs[:, col]:
        ax.axis('off')
fig.suptitle('DINOv2 (ViT-B/14) patch features: first three principal components (per image) as RGB')
savefig('ssl_dino_pca')
plt.show()

## Correspondences from self-supervised features

The two photos below were taken from very different angles: SIFT with the ratio test finds only a few reliable matches (Lecture 5). DINOv2 features are much more invariant. We match patches by *mutual nearest neighbours* in feature space (cosine similarity) and, as for SIFT, keep the most *distinctive* matches: those whose best match is much more similar than the second best (this removes the sky, where every patch looks like every other). The matches are coarse (one per 14x14 patch, after resizing).

In [ ]:
fa, fb = features['image_cc_1'], features['image_cc_3']
A, B = fa.reshape(-1, fa.shape[-1]), fb.reshape(-1, fb.shape[-1])
similarity = A @ B.T
nn_ab, nn_ba = similarity.argmax(1), similarity.argmax(0)
mutual = np.where(nn_ba[nn_ab] == np.arange(len(A)))[0]
top2 = np.sort(similarity, axis=1)[:, -2:]
distinctiveness = top2[:, 1] - top2[:, 0]          # margin between the best and the second-best match
best = mutual[np.argsort(-distinctiveness[mutual])[:60]]
print(f'{len(mutual)} mutual nearest neighbours, showing the 60 most distinctive')

def patch_center(index, f, image):
    r, c = np.unravel_index(index, f.shape[:2])
    return np.array([(c + 0.5) * image.shape[1] / f.shape[1], (r + 0.5) * image.shape[0] / f.shape[0]])

# for comparison: SIFT with the ratio test
sift = cv2.SIFT_create()
ga, gb = [cv2.cvtColor(images[n], cv2.COLOR_RGB2GRAY) for n in ['image_cc_1', 'image_cc_3']]
ka, da = sift.detectAndCompute(ga, None)
kb, db = sift.detectAndCompute(gb, None)
sift_matches = [m for m, n in cv2.BFMatcher().knnMatch(da, db, k=2) if m.distance < 0.7 * n.distance]
print(f'SIFT: {len(sift_matches)} matches pass the ratio test (0.7)')

ia, ib = images['image_cc_1'], images['image_cc_3']
canvas = np.concatenate([ia, ib], axis=1)
fig, ax = plt.subplots(figsize=(20, 7))
ax.imshow(canvas)
colors = plt.cm.hsv(np.linspace(0, 1, len(best), endpoint=False))
for i, color in zip(best, colors):
    pa, pb = patch_center(i, fa, ia), patch_center(nn_ab[i], fb, ib) + [ia.shape[1], 0]
    ax.plot([pa[0], pb[0]], [pa[1], pb[1]], '-', color=color, linewidth=1)
    ax.scatter(*np.stack([pa, pb]).T, color=color, s=15)
ax.set_title(f'DINOv2 mutual nearest neighbours (60 most distinctive); SIFT + ratio test finds {len(sift_matches)} matches')
ax.axis('off')
savefig('ssl_dino_correspondences')
plt.show()

## CLIP vs. SigLIP

Both models embed images and texts into a joint space and score image-text pairs by cosine similarity. They differ in the training loss:

* **CLIP** uses a *softmax* over all texts in the batch: the probabilities of all captions for an image sum to one, so captions compete.
* **SigLIP** uses a *sigmoid* per image-text pair: every caption gets an independent probability of being a match.

For an image that fits several captions, CLIP has to spread or pick; SigLIP can say yes to all of them.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, AutoProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch16')
siglip = AutoModel.from_pretrained('google/siglip-base-patch16-224').to(device).eval()
siglip_processor = AutoProcessor.from_pretrained('google/siglip-base-patch16-224')

def clip_probs(image, texts):
    inputs = clip_processor(text=texts, images=image, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        return clip(**inputs).logits_per_image.softmax(-1)[0].cpu().numpy()

def siglip_probs(image, texts):
    inputs = siglip_processor(text=texts, images=image, return_tensors='pt', padding='max_length', max_length=64).to(device)
    with torch.no_grad():
        return torch.sigmoid(siglip(**inputs).logits_per_image)[0].cpu().numpy()

captions = ['a photo of a dog.', 'a photo of a dog in the snow.', 'a photo of a dog next to a lantern.',
            'a photo of snow.', 'a photo of a lantern.', 'a photo of a cat.', 'a photo of a beach.']
image = images['image_dog']
p_clip, p_siglip = clip_probs(image, captions), siglip_probs(image, captions)

fig, axs = plt.subplots(1, 3, figsize=(22, 5), gridspec_kw={'width_ratios': [1, 1.3, 1.3]})
axs[0].imshow(image)
axs[0].axis('off')
for ax, p, title in [(axs[1], p_clip, f'CLIP: softmax over captions (sums to {p_clip.sum():.2f})'),
                     (axs[2], p_siglip, f'SigLIP: independent sigmoid per caption (sums to {p_siglip.sum():.2f})')]:
    ax.barh(range(len(captions)), p, color='tab:blue')
    ax.set_yticks(range(len(captions)))
    ax.set_yticklabels(captions)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_title(title)
    for i, v in enumerate(p):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center')
plt.tight_layout()
savefig('ssl_clip_vs_siglip')
plt.show()

## Models learn to read

CLIP is trained on images with their alt-text from the web, where text in the image often matches the caption. So it learned to read, and text in an image can override what it shows: a *typographic attack*. We put a small sticker with a word next to the dog and classify the image zero-shot.

In [ ]:
def sticker(image, text):
    image = image.copy()
    h, w = image.shape[:2]
    cv2.rectangle(image, (int(0.6 * w), int(0.55 * h)), (int(0.95 * w), int(0.8 * h)), (255, 255, 255), -1)
    cv2.putText(image, text, (int(0.62 * w), int(0.73 * h)), cv2.FONT_HERSHEY_DUPLEX, 4 * w / 1280, (0, 0, 0), int(8 * w / 1280))
    return image

labels = ['dog', 'cat', 'fox', 'wolf', 'lantern', 'car']
prompts = [f'a photo of a {l}.' for l in labels]
fig, axs = plt.subplots(2, 4, figsize=(24, 9), gridspec_kw={'height_ratios': [1.6, 1]})
for col, text in enumerate([None, 'CAT', 'FOX', 'iPod']):
    attacked = image if text is None else sticker(image, text)
    p = clip_probs(attacked, prompts)
    axs[0, col].imshow(attacked)
    axs[0, col].set_title('original' if text is None else f'with a "{text}" sticker')
    axs[0, col].axis('off')
    axs[1, col].bar(labels, p, color=['tab:red' if i == p.argmax() else 'tab:blue' for i in range(len(labels))])
    axs[1, col].set_ylim(0, 1)
    axs[1, col].set_title(f'CLIP zero-shot: {labels[p.argmax()]} ({p.max():.2f})')
savefig('ssl_typographic_attack')
plt.show()